- Dataset & Dataloader Class

- Problems
    1) Memory inefficient due to batch gradient decent.
    2) Not Better Convergence due to batch gradient decent.

    - Solution:
        - Using batches (mini batch gradient descent) of data to train the model.

- The Dataset and Dataloader Classes
    - Dataset and Dataloader are core abstraction in PyTorch that decouple how you define your data from how you efficiently iterate over it in training loop. This means we sperate data load and data training process.
    - Consider we have csv data file having 10 rows and batch size is 2 so toatal batches = 10/2 = 5. 
        - We have to create custom dataset class which inherit from `CustomDataset(Dataset)`
    - Dataset Class:
        - The dataset class is essentially a blueprint when you create custom dataset, you decide how data is loaded and returned.
        - It defines:
            - `__init__()` which tells how data should be loaded.
            - `__len__()` which returns the total number of samples.
            - `__getitem__(index)` which returns the data (and labels) at given index

    - DataLoader Class:
        - The DataLoader wraps a Dataset and handles batching, shuffling and parallel loading.
    - DataLoader Control Flow:
        - At the start of each epoch, the DataLoader (if `shuffle=True`) shuffle indices(using a sampler)
        - It divides the indices into chunck of batch size.
        - For each index in the chunk, data samples are fetched from the Dataset object
        - The samples are then collected and combined into a batch (using collate function)
        - The batch is returned to the main training loop


In [52]:
from sklearn.datasets import make_classification
import torch

In [53]:
# step1: Create a synthetic classification dataset using sklearn
X , y = make_classification(
    n_samples= 10,          # number of samples
    n_features= 2,          # number of features
    n_informative=2,        # number of informative features
    n_redundant= 0,         # number of redundant fetures
    n_classes=2,            # number of classes
    random_state=42         # for reproducibiliy

)

In [54]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [55]:
X.shape

(10, 2)

In [56]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [57]:
y.shape

(10,)

In [58]:
# covert to tensor
X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y,dtype=torch.float32)

In [59]:
X

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])

In [60]:
X.shape

torch.Size([10, 2])

In [61]:
y

tensor([1., 0., 0., 0., 0., 1., 1., 1., 1., 0.])

In [62]:
y.shape

torch.Size([10])

In [63]:
from torch.utils.data import Dataset, DataLoader

In [64]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [65]:
dataset = CustomDataset(X,y)

In [66]:
len(dataset)

10

In [67]:
dataset[0]

(tensor([ 1.0683, -0.9701]), tensor(1.))

In [ ]:
dataloder = DataLoader(dataset,batch_size=2,shuffle=True,num_workers=5,collate_fn=)

In [71]:
for batch_features, batch_labels in dataloder:
    print(batch_features)
    print(batch_labels)
    print("-"*70)

tensor([[-0.7206, -0.9606],
        [ 1.7273, -1.1858]])
tensor([0., 1.])
----------------------------------------------------------------------
tensor([[-0.5872, -1.9717],
        [-1.1402, -0.8388]])
tensor([0., 0.])
----------------------------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-1.9629, -0.9923]])
tensor([0., 0.])
----------------------------------------------------------------------
tensor([[ 1.8997,  0.8344],
        [ 1.0683, -0.9701]])
tensor([1., 1.])
----------------------------------------------------------------------
tensor([[-0.9382, -0.5430],
        [ 1.7774,  1.5116]])
tensor([1., 1.])
----------------------------------------------------------------------


- In PyTorch, the sampler in the DataLoader determines the strategy for selecting the dataset during data loading. It controls how indices of the dataset are drawn from each batch.

- Types of Samplers
    1) SequentialSampler:
        - Samples element sequentially, in the order they appear in the dataset.
        - Default when `shuffle=false`

    2) RandomSampler:
        - Samples elements randomly without replacement.
        - Defaukt when `shuffle=True`

- Collate Function (`collate_fn`):
    - The `collare-fn` in PyTorch DataLoader is a function that specifies how to combine a list of samples from a dataset into single batch. By default, the DataLoader uses a simple batch collation mechanism, but `collate_fn` allows you to customize how the data should be processed and batched.